# Active Learning NEP

In [ ]:
import os
import numpy as np
import copy as cp
import json
from ase.io import read, write
from ase.visualize import view
#import calorine
from calorine.nep import setup_training, get_descriptors
import phonopy
from hiphive.structure_generation import generate_phonon_rattled_structures
import matplotlib.pyplot as plt
from src.latexfig import LatexFigure
from src.phononASE import phonopy_to_ase

In the following, we go through each step of the active learning scheme.

In [ ]:
from src.activeNEP import ActiveLearningNEP
NEP = ActiveLearningNEP('results/ALnep')

In [ ]:
from src.calculators import copy_calc_results

In [ ]:
from ase.build import sort

In [ ]:
sort(NEP.train_data[-1])

In [ ]:
# Sort atoms by alphabetical order of chemical symbols to ensure consistency in descriptor calculation and active set selection

train_data = []
test_data = []
for atoms in NEP.train_data:
    try:
        sorted_atoms = copy_calc_results(atoms, sort=True)
    except RuntimeError:
        sorted_atoms = sort(atoms)
    train_data.append(sorted_atoms)

for atoms in NEP.test_data:
    try:
        sorted_atoms = copy_calc_results(atoms, sort=True)
    except RuntimeError:
        sorted_atoms = sort(atoms)
    test_data.append(sorted_atoms)


In [ ]:
def shift_energies(data):
    # Attempt to read energies from energies.json, which should have been generated by the DFT calculations
    try:
        with open(os.path.join(NEP.run_dir, 'energies.json'), 'r') as f:
            energies = json.load(f)

    except FileNotFoundError:
        energies = None
        print("Atomic energies file (energies.json) not found in directory.")
        print("It is highly recommended to have the atomic energies of the constituent elements for better NEP training.", flush=True)

    # Shift the energies of the structures by the sum of the energies of the constituent atoms, if energies are available
    for atoms in data:
        elements = atoms.get_chemical_symbols()
        if energies is not None:
            atoms.calc.results['energy'] += sum(energies[element] for element in elements)

In [ ]:
#shift_energies(NEP.train_data)

In [ ]:
write(os.path.join(NEP.run_dir, f"train.xyz"), train_data)
write(os.path.join(NEP.run_dir, f"test.xyz"), test_data)

## Active Learning Loop

### 0. Data generation

If the dataset has not yet been generated, it is done by phonon ratteling & strain.

This requires .yaml files for constructing force constants matrix and obtaining phonon modes.

In [ ]:
# Prepare dataset by phonon ratteling
NEP.prepare_dataset()

### 1. DFT labeling

Now DFT is performed on all dataset structures in train.xyz and test.xyz

In [ ]:
# Run DFT calculations on the generated structures
#NEP.run_DFT()      # run on HPC CPU node (sylg.fysik.dtu.dk)

### 2. Training NEP

The structures in train.xyz and test.xyz are copied to the iteration folder.

During this process, energies are shifted by atomic energies and atoms ordered alphabetically.

Furthermore, the necesassy nep.in file is created based on the parameters bellow.

In [ ]:
# Define parameters for training the NEP model
params = dict(cutoff=[8, 4],
              neuron=30,
              generation=100000,
              batch=1000000)
NEP.setup_nep(**params)
#NEP.train_nep()    # run on HPC GPU node (surt.fysik.dtu.dk or sara.fysik.dtu.dk)

In [ ]:
# Count number of atoms in the training dataset

sum(len(struct) for struct in NEP.train_data)

In [ ]:
view(NEP.train_data)

We can inspect training to see, how the loss and RMSE values change with generation.

In [ ]:
import numpy as np
import pandas as pd
from ase.units import GPa
from calorine.nep import get_parity_data, read_loss, read_structures
from matplotlib import pyplot as plt
from pandas import DataFrame, concat as pd_concat
from sklearn.metrics import r2_score, mean_squared_error

In [ ]:
def plot_loss(iter=1):
    loss = read_loss(os.path.join(NEP.run_dir, f'iteration_{iter}', 'loss.out'))

    lf = LatexFigure()
    fig, axes = lf.create(AR=0.25, subplots=(2, 1), sharex=True)

    ax = axes[0]
    ax.set_ylabel('Loss')
    ax.plot(loss.total_loss, label='total')
    ax.plot(loss.L2, label='$l_2$')
    ax.plot(loss.L1, label='$l_1$')
    ax.set_yscale('log')
    ax.legend()

    ax = axes[1]
    ax.plot(loss.RMSE_F_train, label='forces (eV/Å)')
    ax.plot(loss.RMSE_V_train, label='virial (eV/atom)')
    ax.plot(loss.RMSE_E_train, label='energy (eV/atom)')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Generation')
    ax.set_ylabel('RMSE')
    ax.legend()

    #fig.subplots_adjust(hspace=0)
    fig.set_constrained_layout_pads(hspace=0.0, h_pad=0.0)
    plt.show()

def plot_parity(iter=1):
    training_structures, _ = read_structures(os.path.join(NEP.run_dir, f'iteration_{iter}'))

    units = dict(
        energy='eV/atom',
        force='eV/Å',
        virial='eV/atom',
        stress='GPa',
    )
    # Make a 2x2 grid of parity plots for energy, force, virial, and stress
    lf = LatexFigure()
    fig, axes = lf.create(width=0.8, AR=0.9, subplots=(2, 2))
    kwargs = dict(alpha=0.2, s=0.3)
    axes = axes.flatten()

    # Loop over the properties and units, get the parity data, calculate R2 and RMSE, and plot the parity plots
    for icol, (prop, unit) in enumerate(units.items()):
        df = get_parity_data(training_structures, prop, flatten=True)
        R2 = r2_score(df.target, df.predicted)
        rmse = np.sqrt(mean_squared_error(df.target, df.predicted))

        ax = axes[icol]
        ax.set_xlabel(f'Target {prop} ({unit})')
        ax.set_ylabel(f'Predicted ({unit})')
        # Plot line x = y for reference
        ax.plot([df.target.min(), df.target.max()], [df.target.min(), df.target.max()], 'k-', lw=0.8, alpha=0.2)
        ax.scatter(df.target, df.predicted, **kwargs)
        ax.set_aspect('equal')
        mod_unit = unit.replace('eV', 'meV').replace('GPa', 'MPa')
        ax.text(0.1, 0.75, f'{1e3*rmse:.1f} {mod_unit}\n' + '$R^2= $' + f' {R2:.5f}', transform=ax.transAxes)
    fig.align_labels()
    plt.show()

In [ ]:
plot_loss()

We can also inspect predicted vs target values for energy, force, viral and stress.

In [ ]:
plot_parity()

### 3. Build active set

With the NEP model trained, we need to build an active set to calculate extrapolation grades.

In [ ]:
# Build Active Set (.asi) with MaxVol
NEP.build_active_set()

In [ ]:
#view(NEP.active_set_struct)

Now that we have the active set, we can calculate the extrapolation grade of all atoms.

In [ ]:
NEP.assign_gamma(NEP.train_data)

In [ ]:
def plot_gamma_model(structures, per_struct=True):
    import math
    import numpy as np
    if per_struct:
        gammas = np.array([max(structures[i].arrays["gamma"]) for i in range(len(structures))])
    else:
        gammas = np.array([gamma for struct in structures for gamma in struct.arrays["gamma"]])
    # Round gamma values to 5 decimals
    gammas = np.round(gammas, 5)
    
    print(f"Gamma range: {gammas.min()} - {gammas.max()}")

    g_min = math.floor(np.min(gammas) * 2) / 2
    g_max = math.ceil(np.max(gammas) * 2) / 2

    lf = LatexFigure()
    fig, axes = lf.create(AR=0.6, style='bands')

    for ax in axes:
        #ax.set_title('Distribution of Extrapolation Grade (gamma)')

        ax.hist(gammas, range=(g_min, g_max), bins=100,
                color='steelblue', edgecolor='black', lw=0.8, alpha=1, density=False)

        ax.set_xlabel('Extrapolation Grade ($\\gamma$)')
        if per_struct:
            ax.set_ylabel('Structures')
        else:
            ax.set_ylabel('Environments')

        ax.set_xlim(g_min, g_max)

        ax.set_yscale('log')

    plt.show()

In [ ]:
plot_gamma_model(NEP.train_data, per_struct=False)

As expected, all structures have extrapolation bellow 1.

We now investigate strained structures

In [ ]:
# Apply some strain to the active set structures and re-assign gamma to see how the extrapolation grade changes
NEP.build_active_set()
for atoms in NEP.active_set_struct:
    NEP._strain_structure(atoms, 5)


In [ ]:
NEP.assign_gamma(NEP.active_set_struct)
plot_gamma_model(NEP.active_set_struct, per_struct=False)

We can now test how resilient the dataset is to larger supercells/systems.

In [ ]:
#sc_structures = [NEP.train_data[i].repeat((2, 2, 1)) for i in range(len(NEP.train_data))]

In [ ]:
NEP.assign_gamma(sc_structures)

In [ ]:
plot_gamma_model(sc_structures)

These are all bellow 2, so considered a safe extrapolation.

What if we try to make thicker slabs than the model has been trained on?

In [ ]:
from src.structure import Perovskite
slab_structures = [Perovskite('BaTiO3', bulk=False, dslab=(1.5+i)).atoms.repeat((2, 2, 1)) for i in range(2, 20)]

In [ ]:
slab_descriptors = NEP._calculate_descriptors(slab_structures)

In [ ]:
NEP.assign_gamma(slab_structures, slab_descriptors)
plot_gamma_model(slab_structures)

Does not change at all. Note strain on lattice, should use relaxed slabs for lower gamma.

Now additional phonon ratteling.

In [ ]:
import phonopy as ph
from src.phononASE import phonon_to_atoms
phonon = ph.load('results/ALnep/bulk.yaml')

# Produce the force constants
phonon.produce_force_constants()
# Get the force constants matrix
fc = phonon.force_constants

# Convert the phonon object to an ASE Atoms object
atoms = phonon_to_atoms(phonon, cell='super')
rattled_structures = NEP._get_rattled_structures(atoms, fc, 500, 1000)

In [ ]:
NEP.assign_gamma(rattled_structures)
plot_gamma_model(rattled_structures)

### 4. MD exploration

We now run MD to explore for new configurations that have high extrapolation grade.

In [ ]:
# Define parameters for running MD simulations with the trained NEP model
md_params = dict(dt=1,
                 n_steps=2*1e5,
                 n_dump=1000,
                 temperatures=[100, 200, 300, 400, 500, 600])
NEP.setup_MD(**md_params)
#NEP.run_MD()       # run on HPC GPU node (surt.fysik.dtu.dk or sara.fysik.dtu.dk)

In [ ]:
NEP.collect_MD_structures()

In [ ]:
md_structures = read(os.path.join(NEP.iter_dir, 'md/md_structures.xyz'), index=':')

In [ ]:
test = read('results/ALnep/iteration_1/md/temp/run_003/dump.xyz', index=':')
len(test)

In [ ]:
view(test)

In [ ]:
NEP.assign_gamma(test)

In [ ]:
plot_gamma_model(test, per_struct=False)

In [ ]:
# Collect structure from different MD runs
#NEP.collect_MD_structures()
md_structures = []
for temp in ['100K', '200K', '300K', '400K', '500K', '600K']:
    md_structures.extend(read(os.path.join(NEP.iter_dir, f'md_old4/Ba4O20Ti8/{temp}/dump.xyz'), ':'))
#md_structures
#md_test = md_test[-1000:]

In [ ]:
md_structures = read(os.path.join(NEP.iter_dir, 'md/md_structures.xyz'), index=':')

In [ ]:
from ase.visualize import view
view(md_structures)

In [ ]:
NEP.assign_gamma(md_structures)
plot_gamma_model(md_structures, per_struct=False)

In [ ]:
#from src.structure import wrap_to_reference
def wrap_continuous(traj, ref_atoms):
    wrapped = []

    #prev = ref_atoms.copy()
    #wrapped.append(prev)

    #cell = ref_atoms.get_cell()
    ref_scaled = ref_atoms.get_scaled_positions()

    for atoms in traj:
        new_atoms = atoms.copy()

        scaled = atoms.get_scaled_positions()

        delta = scaled - ref_scaled
        delta -= np.floor(delta + 0.5)  # wrap into [-0.5, 0.5]

        new_scaled = ref_scaled + delta

        new_atoms.set_scaled_positions(new_scaled)

        wrapped.append(new_atoms)

        # update reference
        ref_scaled = new_scaled

    return wrapped

In [ ]:
with open(os.path.join(NEP.iter_dir, 'md/Ba4O20Ti8/100 K/run.in'), 'r') as f:
    run_in = f.read()

In [ ]:
import numpy as np

def load_thermo(thermo_file, run_file):
    import re

    with open(run_file, 'r') as f:
        run_in = f.read()

    time_step = int(re.findall(r'time_step\s+(\d+\.?\d*)', run_in)[0])
    dump_thermo = int(re.findall(r'dump_thermo\s+(\d+)', run_in)[0])
    n_steps = int(re.findall(r'run\s+(\d+)', run_in)[0])*2

    t = np.linspace(0, n_steps*time_step, n_steps//dump_thermo)
    data = np.loadtxt(thermo_file)

    def _get_lattice_constants(cell):
        a = np.linalg.norm(cell[:, 0], axis=1)
        b = np.linalg.norm(cell[:, 1], axis=1)
        c = np.linalg.norm(cell[:, 2], axis=1)
        return a, b, c

    return {
        "t": t,             # time in fs
        "T": data[:, 0],    # temperature in K
        "K": data[:, 1],    # kinetic energy in eV
        "U": data[:, 2],    # potential energy in eV
        "Pxx": data[:, 3],  # pressure components in GPa
        "Pyy": data[:, 4],  
        "Pzz": data[:, 5],
        "Pyz": data[:, 6],
        "Pxz": data[:, 7],
        "Pxy": data[:, 8],
        "cell": _get_lattice_constants(data[:, 9:].reshape(-1, 3, 3))
    }

In [ ]:
# read thermo.out and extract temperature and energy
thermo_file = os.path.join(NEP.iter_dir, 'md/tests/1/thermo.out')
run_file = os.path.join(NEP.iter_dir, 'md/tests/1/run.in')
thermo = load_thermo(thermo_file, run_file)

In [ ]:
plt.plot(thermo['t'], thermo['T'])

In [ ]:
plt.plot(thermo['t']/1e3, thermo['cell'][0], label='a')
plt.plot(thermo['t']/1e3, thermo['cell'][1], label='b')
plt.plot(thermo['t']/1e3, thermo['cell'][2]-40, label='c')
plt.xlabel('Time (ps)')
plt.ylabel('Lattice Constant (Å)')
plt.legend()
plt.show()

In [ ]:
a= np.array([])
b = np.array([])
c = np.array([])
T = np.array([])

for temp in ['100 K', '200 K', '300 K', '400 K', '500 K', '600 K']:
    run_file = os.path.join(NEP.iter_dir, f'md/Ba4O20Ti8/{temp}/run.in')
    thermo_file = os.path.join(NEP.iter_dir, f'md/Ba4O20Ti8/{temp}/thermo.out')
    thermo = load_thermo(thermo_file, run_file)
    T = np.append(T, thermo['T'])
    a = np.append(a, thermo['cell'][0])
    b = np.append(b, thermo['cell'][1])
    c = np.append(c, thermo['cell'][2]-40)

In [ ]:
plt.plot(T, a, '.', label='a')
plt.plot(T, b, '.', label='b')
#plt.plot(T, c, '.', label='c')
plt.xlabel('Temperature (K)')
plt.ylabel('Lattice Constant (Å)')
plt.title('Lattice Constants vs. Temperature')
plt.legend()
plt.show()

In [ ]:
md_test = read(os.path.join(NEP.iter_dir, f'md/Ba4O20Ti8/100 K/dump.xyz'), ':')

In [ ]:

def get_lattice_constants(atoms):
    positions = atoms.get_positions()
    a = np.max(positions[:, 0]) - np.min(positions[:, 0])
    b = np.max(positions[:, 1]) - np.min(positions[:, 1])
    c = np.max(positions[:, 2]) - np.min(positions[:, 2])
    return a, b, c

In [ ]:
plt.plot(T, label='Temperature (K)')
# Plot straight line from T0 to T1  
plt.plot([0, len(T)//2], [20, 1000], 'k--', label='Target Temperature')
plt.plot([len(T)//2, len(T)], [1000, 20], 'k--')
plt.xlabel('Time step')
plt.ylabel('Temperature (K)')
plt.legend()
plt.show()

In [ ]:
def compute_volume(cell):
    return np.abs(np.linalg.det(cell))

V = np.array([compute_volume(c) for c in cell])

plt.plot(T, V, '.', alpha=0.5)
plt.xlabel("Temperature (K)")
plt.ylabel("Volume (Å³)")
plt.title("Thermal expansion")
plt.show()

In [ ]:
kB = 8.617333262e-5  # eV/K

E = thermo["K"] + thermo["U"]

def compute_heat_capacity(T, E, window=200):
    Cv = []
    T_mid = []

    for i in range(len(T) - window):
        Ei = E[i:i+window]
        Ti = np.mean(T[i:i+window])

        varE = np.var(Ei)
        Cv.append(varE / (kB * Ti**2))
        T_mid.append(Ti)

    return np.array(T_mid), np.array(Cv)

T_mid, Cv = compute_heat_capacity(T, E)

plt.plot(T_mid, Cv)
plt.xlabel("Temperature (K)")
plt.ylabel("Heat capacity (arb.)")
plt.title("Heat capacity vs temperature")
plt.show()

In [ ]:
Pxx = thermo["Pxx"]
Pyy = thermo["Pyy"]
Pzz = thermo["Pzz"]

anisotropy = np.std(np.vstack([Pxx, Pyy, Pzz]), axis=0)

plt.plot(T, anisotropy, '.', alpha=0.5)
plt.xlabel("Temperature (K)")
plt.ylabel("Pressure anisotropy (GPa)")
plt.title("Symmetry breaking signal")
plt.show()

### 5. Structure filtering

To not consider all structures obtained from MD, we can calculate extrapolation grade and filter.

All structures containing no atoms/environments with gamma > gamma_th are removed.

In [ ]:
md_structures = read(os.path.join(NEP.iter_dir, 'md/md_structures.xyz'), index=':')

In [ ]:
NEP.assign_gamma(md_structures)
plot_gamma_model(md_structures, per_struct=False)

In [ ]:
# Load the collected structures and filter them based on the gamma threshold
#structures = read(os.path.join(NEP.iter_dir, "md/md_structures.xyz"), index=":")
NEP.filter_structures(md_structures, gamma_th=2)

In [ ]:
def run_MD_sim(atoms, temp=300, n_steps=5000):

    from calorine.calculators import CPUNEP
    from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
    from ase.md.langevin import Langevin
    from ase import units

    atoms.calc = CPUNEP(os.path.join(NEP.iter_dir, 'nep.txt'))

    MaxwellBoltzmannDistribution(atoms, temperature_K=temp)

    # 4. Set up MD (Langevin thermostat)
    dyn = Langevin(
        atoms,
        timestep=1.0 * units.fs,
        temperature_K=temp,
        friction=0.01
    )

    dyn.attach(lambda: write('results/test/md/md.xyz', atoms, append=True), interval=100)
    dyn.run(n_steps)

In [ ]:
for i in range(15):
    atoms = read(os.path.join(NEP.iter_dir, f"md/run_{i+1:03d}/model.xyz"))
    run_MD_sim(atoms, temp=700, n_steps=10000)


In [ ]:
md_structures = read("results/test/md/md.xyz", index=":")

In [ ]:
#view(md_structures)

In [ ]:
md_descriptors = NEP._calculate_descriptors(md_structures)

In [ ]:
NEP.assign_gamma(md_structures, md_descriptors)
plot_gamma_model(md_structures)

### 6. Diversity selection

Since we do not wish to perform DFT on all high-gamma structures, diversity selection is performed.

This ensures that only structures with atoms contributing with new basis vectors in the active set are included.

By doing this, only the most diverse environments are included.

In [ ]:
# Read high gamma structures and select based on diversity (MaxVol)
structures = read(os.path.join(NEP.iter_dir, "large_gamma.xyz"), index=":")
#len(structures)
NEP.select_structures(structures, batch_size=40)


In [ ]:
selected_structures = read(os.path.join(NEP.iter_dir, "newdata.xyz"), index=':')

In [ ]:
view(selected_structures)

In [ ]:
len(selected_structures)

In [ ]:
NEP.update_dataset()

In [ ]:
def plot_dF_gamma(structures, per_structure=False):

    dFs = []
    gammas = []
    from calorine.calculators import CPUNEP

    for structure in structures:
        try:
            F_DFT = structure.get_forces().copy()
        except RuntimeError:
            continue
        calculator = CPUNEP(os.path.join(NEP.iter_dir, 'nep.txt'))
        structure.calc = calculator
        F_ML = structure.get_forces()
        # Calculate maximum force error per atom in the structure
        dF = np.max(np.abs(F_DFT - F_ML), axis=1)
        gamma = structure.arrays["gamma"]
        
        if per_structure:
            gamma = np.max(gamma)
            dF = np.max(dF)

        dFs.append(dF)
        gammas.append(gamma)
    
    # Plot force error vs gamma
    lf = LatexFigure()
    fig, axes = lf.create()

    for ax in axes:
        ax.scatter(gammas, dFs, alpha=0.5)
        ax.set_xlabel('$\\gamma$')
        ax.set_ylabel('|dF| (eV/Å)')
        ax.set_xscale('log')
        ax.set_yscale('log')
        #ax.set_xlim(1e-1, 1e2)
        #ax.set_ylim(1e-3, 1e1)


In [ ]:
import phonopy as ph
from src.phononASE import phonon_to_atoms
phonon = ph.load('results/MLtest/BaTiO3.yaml')

# Produce the force constants
phonon.produce_force_constants()
# Get the force constants matrix
fc = phonon.force_constants

# Convert the phonon object to an ASE Atoms object
atoms = phonon_to_atoms(phonon, cell='super')
new_structures = NEP._get_rattled_structures(atoms, fc, 300, 1000)

In [ ]:
new_structures = [NEP.train_data[i].repeat((2, 2, 2)) for i in range(len(NEP.train_data))]

In [ ]:
from ase.visualize import view
view(new_structures)

In [ ]:
NEP.filter_structures(new_structures, gamma_th=2)

In [ ]:
filtered_structures = read('results/MLtest/iteration_1/large_gamma.xyz', index=':')

In [ ]:
NEP.select_structures(filtered_structures)

In [ ]:
selected_structures = read('results/MLtest/iteration_1/newdata.xyz', index=':')

In [ ]:
from ase.visualize import view
view(selected_structures[0])

In [ ]:
selected_structures[0].arrays["gamma"]

In [ ]:
np.array(selected_structures[0].get_chemical_symbols())

In [ ]:
view(new_structures[0])

In [ ]:
view(new_structures)

In [ ]:
view(new_structures[0])

In [ ]:
import numpy as np
#gamma_model = np.array([max(NEP.train_data[i].arrays["gamma"]) for i in range(len(NEP.train_data))])
#gamma_new_structures = np.array([max(new_structures[i].arrays["gamma"]) for i in range(len(new_structures))])
gamma_filtered_structures = np.array([max(filtered_structures[i].arrays["gamma"]) for i in range(len(filtered_structures))])
gamma_selected_structures = np.array([max(selected_structures[i].arrays["gamma"]) for i in range(len(selected_structures))])
gamma_min = min(gamma_filtered_structures.min(), gamma_selected_structures.min())
gamma_max = max(gamma_filtered_structures.max(), gamma_selected_structures.max())
print(f'Min extrapolation grade across structures: {gamma_min}')
print(f'Max extrapolation grade across structures: {gamma_max}')

In [ ]:
# Plot distribution of gamma
import matplotlib.pyplot as plt
import math
lf = LatexFigure()

g_min = math.floor(gamma_min * 2) / 2
g_max = math.ceil(gamma_max * 2) / 2

fig, axes = lf.create(AR=0.6)

# Plot vertical line at gamma = 2
axes[0].axvline(x=2, color='black', linestyle='--', label='$\\gamma = 2$')

axes[0].hist(gamma_filtered_structures, label='Filtered Structures', range=(g_min, g_max), bins=101,
             color='steelblue', edgecolor='black', lw=0.8, alpha=1, density=False)
#axes[1].hist(gamma_filtered_structures, label='Filtered Structures', range=(0, g_max), bins=101,
#             color='green', edgecolor='black', alpha=1, density=False)
axes[0].hist(gamma_selected_structures, label='Selected Structures', range=(g_min, g_max), bins=101,
             color='orange', edgecolor='black', lw=0.8, alpha=1, density=False)
#axes[0].hist(gamma_candidates, range=(0, gamma_max), bins=101,
#             color='green', edgecolor='black', alpha=1, density=True)
#plt.hist(get_gamma(descriptors, NEP.active), bins=50, color='orange', edgecolor='black')

axes[0].set_title('Distribution of Extrapolation Grade (gamma)')
axes[0].set_xlim(0, g_max)
axes[0].set_ylabel('Structures')
axes[0].legend()

axes[0].set_xlabel('Extrapolation Grade (gamma)')

#axes[0].grid(True, linestyle='--', alpha=0.5)
plt.show()

In [ ]:
def plot_active_set(A, idx):
    from sklearn.decomposition import PCA
    import matplotlib.pyplot as plt

    pca = PCA(n_components=2)
    A2 = pca.fit_transform(A)

    lf = LatexFigure()
    fig, axes = lf.create()
    # Plot ALL environments as a faint cloud
    axes[0].scatter(
        A2[:, 0], A2[:, 1],
        s=15,
        alpha=0.4,
        color="gray",
        label="All environments"
    )

    # Plot active set clearly
    axes[0].scatter(
        A2[idx, 0], A2[idx, 1],
        s=10,
        marker='x',
        #edgecolor="black",
        color="red",
        label="Active set"
    )
    axes[0].set_title("Active set coverage of descriptor space")
    axes[0].set_xlabel("PC1")
    axes[0].set_ylabel("PC2")
    axes[0].legend()
    #axes[0].tight_layout()
    plt.show()

In [ ]:
def check_active_set_quality(A, idx):
    import numpy as np
    import matplotlib.pyplot as plt

    sub = A[idx]
    inv = np.linalg.pinv(sub)

    # projection coefficients
    C = A @ inv  # (N, M)

    # reconstruction back in descriptor space
    A_proj = C @ sub  # (N, M)

    # relative error per atom
    residual = np.linalg.norm(A - A_proj, axis=1) / (np.linalg.norm(A, axis=1) + 1e-12)

    print(f"Mean reconstruction error: {np.mean(residual):.4e}")
    print(f"Max reconstruction error: {np.max(residual):.4e}")

    plt.hist(residual, bins=50)
    plt.title("Descriptor reconstruction error (active set quality)")
    plt.show()

In [ ]:
plot_active_set(NEP.descriptors, NEP.active_index)
check_active_set_quality(NEP.descriptors, NEP.active_index)

In [ ]:
from ase.io import read
from calorine.calculators import CPUNEP
from calorine.tools import relax_structure, get_force_constants

In [ ]:
from src.structure import Perovskite
perovskite = Perovskite('BaTiO3', N=2)
atoms = perovskite.atoms
calculator = CPUNEP('results/MLtest/iteration_1/nep.txt')
atoms.calc = calculator
relax_structure(atoms, fmax=0.00001)

In [ ]:
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase.md.langevin import Langevin
from ase import units

MaxwellBoltzmannDistribution(atoms, temperature_K=300)

# 4. Set up MD (Langevin thermostat)
dyn = Langevin(
    atoms,
    timestep=1.0 * units.fs,
    temperature_K=300,
    friction=0.01
)

dyn.attach(lambda: write('results/MLtest/md.xyz', atoms, append=True), interval=10)
dyn.run(5000)

In [ ]:
from ase.visualize import view
view(read('results/MLtest/md.xyz', ':'))

## Model predictions

### Phonon dispersions

In [ ]:
from ase.io import read
from calorine.calculators import CPUNEP
from calorine.tools import relax_structure, get_force_constants
from src.structure import Perovskite
from src.structure import check_if_bulk

In [ ]:
from src.phononcalc import get_phonon_dispersion, get_phonon_dos, get_phonon_pdos, order_labels
from matplotlib.ticker import AutoMinorLocator

In [ ]:
def plot_dispersion(phonons, vals, width=1, bulk=True, pDOS=False):
    """Function to plot the phonon dispersion and DOS together.
    Parameters:
    - phonons: List of Phonopy objects containing phonon data.
    - vals: List of labels for each phonon object.
    - width: Width of the figure as a fraction of the total width.
    - pDOS: Whether to plot the projected density of states (PDOS) (default is True).
    - bulk: Boolean indicating if the system is bulk (True) or slab (False).
    Returns:
    - None. The function creates a plot of the phonon dispersion and DOS.
    """

    # Define tickmarks for the x- and y-axis
    E_tickmarks = np.arange(-10, 26, 5)
    # Convert tickmarks to strings with i for negative numbers
    E_ticklabels = []
    for tick in E_tickmarks:
        if tick < 0:
            E_ticklabels.append(f'{abs(tick)}i')
        else:
            E_ticklabels.append(f'{tick}')

    # Define tickmarks for the x- and y-axis
    dos_tickmarks = np.arange(0, 7, 1)

    colors = ["black", "blue", "red", "purple", "orange", "green"]

    # Make a simple figure where graphs are plotted
    lf = LatexFigure()
    fig, axes = lf.create(width=width, AR=0.9, subplots=(1, 2), style='bands', minor=False,
                          sharey='col', gridspec_kw={'width_ratios': [1, 0.4]})
    #fig.set_constrained_layout_pads(wspace=0.01, w_pad=0.01)
    # fig, axes = plt.subplots(1, 2, figsize=(9.6, 5),
    #                          sharey='col', gridspec_kw={'width_ratios': [1, 0.4]})
    
    # Define two axes, one for the band structure and one for the DOS
    ax1, ax2 = axes
    
    #PlotSettings().set_size(fig, width)
    #plt.subplots_adjust(wspace=0.08)
    
    def _plot_disp(ax, phonon, val, col='k', vlines=True):
        # Extract phonon dispersion data
        (dist, X, freq, labels) = get_phonon_dispersion(phonon, bulk)
        dist = np.array(dist)
        dist /= dist[-1][-1]  # Normalize distances to the total length of the path
        X /= X[-1]  # Normalize high symmetry point locations to the total length of the path
        if vlines:
            # Plot vertical lines at symmetry points
            ax.vlines(X, E_tickmarks[0], E_tickmarks[-1], color='0.5', lw=0.8)
        # Plot dashed line at 0
        #ax.axhline(y=0, color='k', linestyle=':')
        # Determine the number of segments between symmetry points and the number of modes
        n_segments = len(freq)
        n_modes = freq[0].shape[1]
        # Loop over all segments and modes and plot everything
        for i in range(n_segments):
            for j in range(n_modes):
                if i == 0 and j == 0:
                    ax.plot(dist[i], freq[i][:, j], color=col, lw=1, label=f'{val}')
                else:
                    ax.plot(dist[i], freq[i][:, j], color=col, lw=1)
        # Set x- and y-ticks
        ax.set_xticks(X, labels)
        ax.set_yticks(E_tickmarks, E_ticklabels)
        # Set x- and y-limits
        ax.set_xlim(X[0], X[-1])
        ax.set_ylim(E_tickmarks[0], E_tickmarks[-1])

    def _plot_dos(ax, phonon, val='DOS', col='k'):
        # Extract total DOS data
        (dosx, dosy) = get_phonon_dos(phonon, bulk)
        # Plot total DOS
        ax.plot(dosx, dosy, lw=1, color=col, label=f'{val}')
        if pDOS:
            ax.fill_between(dosx, dosy, color='lightgray', alpha=0.5)

    def _plot_pdos(ax, phonon):
        atom_colors = {'Ba': 'tab:blue', 'Sr': 'tab:purple',
                       'Ti': 'tab:orange', 'O': 'tab:red'}
        # Extract PDOS data
        (pdosx, pdosy, symbols) = get_phonon_pdos(phonon, bulk)
        # Plot PDOS
        for i in range(pdosx.shape[0]):
            ax.plot(pdosx[i], pdosy, lw=1, color=atom_colors[symbols[i]], label=f'{symbols[i]}')
        # Get all handles and labels
        handles, labels = ax.get_legend_handles_labels()
        # Remove duplicates and sort for the legend
        sorted_handles, sorted_labels = order_labels(symbols, handles, labels)
        # Add legend with duplicates removed and sorted labels
        ax.legend(sorted_handles, sorted_labels, loc='best', fontsize=14)

    # Plot dashed line at Fermi level for both subplots
    ax1.axhline(y=0, color='k', linestyle=':', lw=0.8)
    ax2.axhline(y=0, color='k', linestyle=':', lw=0.8)
    for i, phonon in enumerate(phonons):
        # Plot phonon dispersion and total DOS for PW
        _plot_disp(ax1, phonon, val=vals[i], col=colors[i])
        _plot_dos(ax2, phonon, val=vals[i], col=colors[i])
        if pDOS:
            # Plot PDOS for PW
            _plot_pdos(ax2, phonon)

    # Set x- and y-label
    ax1.set_xlabel('k-points')
    ax1.set_ylabel('Frequency (THz)')
    # Add minor tickmarks to the y-axis
    ax1.yaxis.set_minor_locator(AutoMinorLocator())
    
    ax2.set_xlabel('DOS (1/THz)')
    ax2.legend(loc='upper right')
    # Force x- and y-ticks
    ax2.set_xticks(dos_tickmarks, dos_tickmarks)
    ax2.set_yticks(E_tickmarks, E_ticklabels)
    # Set limits to match
    ax2.set_xlim(dos_tickmarks[0], dos_tickmarks[-1])
    ax2.set_ylim(E_tickmarks[0], E_tickmarks[-1])
    # Hide y-tick labels
    ax2.set_yticklabels([])
    
    # Show figure
    plt.show()

In [ ]:
# Calculate phonons for the bulk unit cell structure using the trained NEP model
perovskite = Perovskite('BaTiO3')
atoms = perovskite.atoms.copy()
NEP.relax_atoms(atoms)
phonon_ML = NEP.calculate_phonon(atoms)
# Load the phonon DFT calculation results from the YAML file
phonon_LCAO = phonopy.load("results/bulk/BaTiO3/0082/phonons/BaTiO3.yaml")
plot_dispersion([phonon_LCAO, phonon_ML], ['LCAO', 'ML'], width=1, bulk=True)

In [ ]:
strains = np.linspace(-0.01, 0.01, 5)

phonons = []
for strain in strains:
    perovskite.set_atoms(atoms)
    perovskite.apply_strain(strain)
    if strain != 0:
        NEP.relax_atoms(perovskite.atoms, strained=True)
    else:
        NEP.relax_atoms(perovskite.atoms)
    phonon = NEP.calculate_phonon(perovskite.atoms)
    phonons.append(phonon)


In [ ]:
plot_dispersion(phonons, np.round(strains, 3), width=1, bulk=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from src.frozenphonon import get_displacement, get_unstable_mode_groups

# ---------- Models ----------

def U_anharm(x, a, b):
    return -0.5 * a * x**2 + 0.25 * b * x**4


# ---------- Utilities ----------

def symmetrize_data(x, y):
    x_sym = np.concatenate((-x[::-1], x))
    y_sym = np.concatenate((y[::-1], y))
    return x_sym, y_sym


def extract_dw_properties(a, b):
    x0 = np.sqrt(a / b)
    barrier = a**2 / (4 * b)
    return x0, barrier


def fit_double_well(x, y):
    #x, y = symmetrize_data(x, y)

    # Shift minimum to zero reference
    y = y - y[np.argmin(np.abs(x))]

    popt, _ = curve_fit(
        U_anharm,
        x,
        y,
        bounds=([0, 0], [np.inf, np.inf])
    )

    a, b = popt
    xfit = np.linspace(min(x), max(x), 1000)
    yfit = U_anharm(xfit, a, b)

    x0, barrier = extract_dw_properties(a, b)

    return x, y, xfit, yfit, x0, barrier

def displace_imaginary_modes(phonon, n_points=101, deg=True):
    """
    """
    # Get current working directory (cwd)
    #cwd = os.getcwd()
    # Unitcell and formula from phonon object
    unitcell = phonon_to_atoms(phonon, cell='unit')
    #formula = unitcell.symbols

    # Dictionary for q-points
    q_dict = {
        'G': [0.0, 0.0, 0.0],
        'X': [0.5, 0.0, 0.0],
        'M': [0.5, 0.5, 0.0],
        'R': [0.5, 0.5, 0.5],
    }

    dd_dict = {
        'G': 0.5/n_points,
        'X': 1.5/n_points,
        'M': 3/n_points,
        'R': 5/n_points,
    }

    results = {}

    for qpoint in q_dict.keys():

        q = q_dict[qpoint]
        dd = dd_dict[qpoint]

        groups, stable = get_unstable_mode_groups(phonon, q)

        if stable:
            print(f"No unstable modes found at {qpoint}")
            continue

        results[qpoint] = {}

        for g_id, group in enumerate(groups):

            modes = group["modes"]
            freq = group["frequency"]

            if not deg:
                modes = [modes[0]]

            results[qpoint][g_id] = {}

            for mode_id, modevec in enumerate(modes):

                modevec_sc, supercell, supercell_matrix = get_displacement(unitcell, q, modevec)

                nx, ny, nz = supercell_matrix.diagonal().astype(int)
                ncells = nx * ny * nz

                calc = CPUNEP(os.path.join(NEP.iter_dir, 'nep.txt'))

                supercell_disp = supercell.copy()
                ref_positions = supercell.positions.copy()
                supercell_disp.calc = calc

                amp = 0
                amplitudes = []
                energies = []
                images = []

                while True:
                    supercell_disp.positions = ref_positions + amp * modevec_sc

                    energy = supercell_disp.get_potential_energy() / ncells

                    amplitudes.append(amp)
                    energies.append(energy)
                    images.append(supercell_disp.copy())

                    amp += dd

                    tol = 100e-3
                    if len(energies) > 1 and energies[-1] - energies[0] > tol:
                        break

                df = pd.DataFrame({
                    'Amplitude': amplitudes,
                    'Energy': energies,
                    'Structure': images
                })
                

                # Find imae corresponding to the minimum energy
                min_idx = np.argmin(energies)
                atoms_min = images[min_idx].copy()
                
                x = df["Amplitude"].to_numpy()
                y = df["Energy"].to_numpy() * 1000  # meV

                x, y, xfit, yfit, x0, barrier = fit_double_well(x, y)

                # store everything
                results[qpoint][g_id][mode_id] = {
                    "df": df,
                    "frequency": freq,
                    "barrier": barrier,
                    "structure": atoms_min
                }

    return results


In [ ]:
from src.phononASE import phonon_to_atoms
import pandas as pd
strains = np.linspace(-0.025, 0.025, 21)

#results = {strain: {} for strain in strains}

perovskite = Perovskite('BaTiO3')
atoms = perovskite.atoms.copy()
NEP.relax_atoms(atoms)

results = {strain: {} for strain in strains}

for strain in strains:
    perovskite.set_atoms(atoms)
    perovskite.apply_strain(strain)

    if strain != 0:
        NEP.relax_atoms(perovskite.atoms, strained=True)
    #else:
        #NEP.relax_atoms(perovskite.atoms)
    
    #latticevecs[qpoint].append(perovskite.atoms.copy().cell.diagonal())

    phonon = NEP.calculate_phonon(perovskite.atoms)
    #phonons.append(phonon)
    results[strain] = displace_imaginary_modes(phonon, n_points=201, deg=False)

# If all frequencies are 0, remove that q-point from the results
#frequencies = {q: freqs for q, freqs in frequencies.items() if any(f != 0 for f in freqs)}
#strainvals = {q: strainvals[q] for q in frequencies.keys()}


In [ ]:
view([results[strains[10]]['G'][0][0]['df']['Structure'][i] for i in range(541)])

In [ ]:
N = 3
mirror = False

#lf = LatexFigure()
#fig, axes = lf.create(subplots=(1, N), AR=2, sharey=True, grid=True)
#color_cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']
for strain in strains:
    res = results[strain]
    lf = LatexFigure()
    N = len(res.keys())
    fig, axes = lf.create(subplots=(1, N), AR=2, sharey=True, grid=True)
    color_cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']

    qpoints = res.keys()

    for i, q in enumerate(qpoints):

        ax = axes[i]
        color_idx = 0

        for g_id, modes in res[q].items():
            for mode_id, data in modes.items():
                df = data["df"]

                x = df["Amplitude"].to_numpy()
                y = df["Energy"].to_numpy() * 1000  # meV

                try:
                    x, y, xfit, yfit, x0, barrier = fit_double_well(x, y)
                except RuntimeError:
                    continue  # skip bad fits

                color = color_cycle[color_idx % len(color_cycle)]
                color_idx += 1

                if mirror:
                    ax.plot(xfit, yfit, '-', color=color)
                else:
                    mask = xfit >= 0
                    ax.plot(xfit[mask], yfit[mask], '-', color=color)

                # Mark origin
                ax.plot(0, 0, 'k.', markersize=4)

                # Annotate physics
                ax.text(
                    0.05,
                    0.9 - 0.1 * color_idx,
                    f"g{g_id} m{mode_id}\n"
                    f"$Q_0$={x0:.2f}\n"
                    f"$\Delta E$={barrier:.1f} meV",
                    transform=ax.transAxes,
                    fontsize=8,
                    color=color
                )

        ax.set_xlabel(r'$Q$ (amu$^{1/2}$Å)')

        if i == 0:
            ax.set_ylabel(r'$\Delta E$ (meV/u.c.)')
    plt.show()


In [ ]:
qpoints = ['G', 'X', 'M']

frequencies = {q: [] for q in qpoints}
barriers    = {q: [] for q in qpoints}
structures  = {q: [] for q in qpoints}

#N = 3

#lf = LatexFigure()
#fig, axes = lf.create(subplots=(1, N), AR=2, sharey=True, grid=True)

for strain in strains:
    res = results[strain]

    #qpoints = res.keys()

    for i, q in enumerate(qpoints):

        #ax = axes[i]
        freqs = []
        bars = []

        try:
            for g_id, modes in res[q].items():
                for mode_id, data in modes.items():

                    freqs.append(data["frequency"])
                    bars.append(data["barrier"])
                    #barrier = data["barrier"]
                    #ax.plot(strain, freq, '.', color='k')
        except KeyError:
            freqs = [0]
            bars = [0]
        
        frequencies[q].append(freqs)
        barriers[q].append(bars)
        structures[q].append(data["structure"])

frequencies = {q: vals for q, vals in frequencies.items() if vals}
barriers = {q: vals for q, vals in barriers.items() if vals}

In [ ]:
"""
perovskite = Perovskite('BaTiO3')
q = 'G'
for i, strain in enumerate(strains):
    perovskite.set_atoms(structures[q][i])

    if strain != 0:
        NEP.relax_atoms(perovskite.atoms, strained=True)
    else:
        NEP.relax_atoms(perovskite.atoms)
"""

In [ ]:
perovskite = Perovskite('BaTiO3')
atoms = perovskite.atoms.copy()
NEP.relax_atoms(atoms)

In [ ]:
perovskite = Perovskite('BaTiO3')
perovskite.set_atoms(structures['G'][10])
NEP.relax_atoms(perovskite.atoms)

In [ ]:
phonon_initial = NEP.calculate_phonon(atoms)
phonon_relaxed = NEP.calculate_phonon(perovskite.atoms)
plot_dispersion([phonon_initial, phonon_relaxed], ['Initial', 'Relaxed'], width=1, bulk=True)

In [ ]:
atoms.cell.diagonal()

In [ ]:

perovskite.atoms.cell.diagonal()

In [ ]:
from ase.visualize import view
view(perovskite.atoms)

In [ ]:
perovskite.atoms

In [ ]:
def plot_imag_freq(strains, y_vals, y_unit='THz'):

    N = len(y_vals.keys())

    lf = LatexFigure()
    fig, axes = lf.create(subplots=(1, N), AR=2, sharey=True, grid=True)
    #fig.set_constrained_layout_pads(wspace=0, w_pad=0)

    def _plot_by_degeneracy(ax, strains, freqs):

        strains = np.array(strains)

        def _group_by_degeneracy(freqs):

            freqs_grouped = []

            for strain, freq in zip(strains, freqs):
                if len(freq) == 1:
                    if strain == 0:
                        # Duplicate
                        pair = [freq[0], freq[0]]
                    else:
                        pair = [freq[0], 0]
                else:
                    pair = freq

                # optional symmetry flip
                if strain < 0:
                    pair = pair[::-1]

                freqs_grouped.append(pair)

            return np.array(freqs_grouped)
        
        try:
            freqs_grouped = np.array(freqs)

            ax.plot(strains * 100, freqs_grouped, '-', lw=1, color='red', label='in-plane')
            ax.plot(strains * 100, freqs_grouped, '.', markersize=2, color='k')
        
        except Exception:
            freqs_grouped = _group_by_degeneracy(freqs)

            y_lower = freqs_grouped[:, 0]
            y_upper = freqs_grouped[:, 1]

            ax.plot(strains * 100, y_lower, '-', lw=1, color='red', label='in-plane')
            ax.plot(strains * 100, y_upper, '-', lw=1, color='blue', label='out-of-plane')
            ax.plot(strains * 100, y_lower, '.', markersize=2, color='k')
            ax.plot(strains * 100, y_upper, '.', markersize=2, color='k')
        
        ax.set_xlim(strains[0] * 100, strains[-1] * 100)
        ax.legend()

    for i, qpoint in enumerate(y_vals.keys()):
        _plot_by_degeneracy(
            axes[i],
            strains,
            y_vals[qpoint]
        )

        axes[i].set_xlabel('Bi-axial strain (%)')

        if qpoint == 'G':
            axes[i].set_title(f'$\Gamma$ point')
        else:
            axes[i].set_title(f'{qpoint} point')

    if y_unit == 'THz':
        axes[0].set_ylabel('Frequency (THz)')
    elif y_unit == 'meV':
        axes[0].set_ylabel('$\\Delta$ Energy (meV/u.c.)')
    #axes[0].set_ylim(-10, -0.1)
    
    plt.show()

In [ ]:
plot_imag_freq(strains, frequencies)

In [ ]:
plot_imag_freq(strains, barriers, y_unit='meV')

In [ ]:
# Calculate phonons for the 1.5 unit cell slab structure using the trained NEP model
atoms = Perovskite('BaTiO3', bulk=False, dslab=1.5).atoms
NEP.relax_atoms(atoms)
phonon_ML = NEP.calculate_phonon(atoms)
# Load the phonon DFT calculation results from the YAML file
phonon_LCAO = phonopy.load("results/slab/BaTiO3/1.5uc/0001/phonons/BaTiO3.yaml")
plot_dispersion([phonon_LCAO, phonon_ML], ['LCAO', 'ML'], width=1, bulk=False)

In [ ]:
# Calculate phonons for the 2.5 unit cell slab structure using the trained NEP model
atoms = Perovskite('BaTiO3', bulk=False, dslab=2.5).atoms
NEP.relax_atoms(atoms)
phonon_ML = NEP.calculate_phonon(atoms)
# Load the phonon DFT calculation results from the YAML file
phonon_LCAO = phonopy.load("results/slab/BaTiO3/2.5uc/0001/phonons/BaTiO3.yaml")
plot_dispersion([phonon_LCAO, phonon_ML], ['LCAO', 'ML'], width=1, bulk=False)

# Tests

In [ ]:
from src.structure import Perovskite
perovskite = Perovskite('BaTiO3')
atoms = perovskite.atoms
calculator = CPUNEP('results/MLtest/nepmodel_split1/nep.txt')
atoms.calc = calculator
relax_structure(atoms, fmax=0.00001)

In [ ]:
atoms.cell

In [ ]:
phonon_ML = get_force_constants(atoms, calculator, [2, 2, 2])

In [ ]:
from src.phononcalc import get_phonon_dispersion, get_phonon_dos, get_phonon_pdos, order_labels
from src.frozenphonon import get_displacement, get_unstable_mode_groups
from src.phononASE import phonon_to_atoms
from src.plotsettings import PlotSettings
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
from ase.io import read, write
from ase.visualize import view
import numpy as np
import pandas as pd

In [ ]:
# Load the phonon calculation results from the YAML file
phonon_LCAO = phonopy.load("results/bulk/BaTiO3/0082/phonons/BaTiO3.yaml")

In [ ]:
def displace_imaginary_modes(phonon, n_points=101, deg=True):
    """
    """
    # Get current working directory (cwd)
    #cwd = os.getcwd()
    # Unitcell and formula from phonon object
    unitcell = phonon_to_atoms(phonon, cell='unit')
    #formula = unitcell.symbols

    # Dictionary for q-points
    q_dict = {
        'G': [0.0, 0.0, 0.0],
        'X': [0.5, 0.0, 0.0],
        'R': [0.5, 0.5, 0.5],
        'M': [0.5, 0.5, 0.0],
    }

    dd_dict = {
        'G': 1/n_points,
        'X': 1.5/n_points,
        'R': 5/n_points,
        'M': 3/n_points,
    }

    results = {}

    for qpoint in q_dict.keys():

        q = q_dict[qpoint]
        dd = dd_dict[qpoint]

        groups, stable = get_unstable_mode_groups(phonon, q)

        if stable:
            print(f"No unstable modes found at {qpoint}")
            continue

        results[qpoint] = {}

        for g_id, group in enumerate(groups):

            modes = group["modes"]
            freq = group["frequency"]

            if not deg:
                modes = [modes[0]]

            results[qpoint][g_id] = {}

            for mode_id, modevec in enumerate(modes):

                modevec_sc, supercell, supercell_matrix = get_displacement(unitcell, q, modevec)

                nx, ny, nz = supercell_matrix.diagonal().astype(int)
                ncells = nx * ny * nz

                calc = CPUNEP(os.path.join(NEP.iter_dir, 'nep.txt'))

                supercell_disp = supercell.copy()
                ref_positions = supercell.positions.copy()
                supercell_disp.calc = calc

                amp = 0
                amplitudes = []
                energies = []

                while True:
                    supercell_disp.positions = ref_positions + amp * modevec_sc

                    energy = supercell_disp.get_potential_energy() / ncells

                    amplitudes.append(amp)
                    energies.append(energy)

                    amp += dd

                    tol = 50e-3
                    if len(energies) > 1 and energies[-1] - energies[0] > tol:
                        break

                df = pd.DataFrame({
                    'Amplitude': amplitudes,
                    'Energy': energies
                })
                
                
                x = df["Amplitude"].to_numpy()
                y = df["Energy"].to_numpy() * 1000  # meV

                x, y, xfit, yfit, x0, barrier = fit_double_well(x, y)

                # store everything
                results[qpoint][g_id][mode_id] = {
                    "df": df,
                    "frequency": freq,
                    "barrier": barrier,
                    "q": q
                }

    return results


In [ ]:
results = displace_imaginary_modes(phonon)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# ---------- Models ----------

def U_anharm(x, a, b):
    return -0.5 * a * x**2 + 0.25 * b * x**4


# ---------- Utilities ----------

def symmetrize_data(x, y):
    x_sym = np.concatenate((-x[::-1], x))
    y_sym = np.concatenate((y[::-1], y))
    return x_sym, y_sym


def extract_dw_properties(a, b):
    x0 = np.sqrt(a / b)
    barrier = a**2 / (4 * b)
    return x0, barrier


def fit_double_well(x, y):
    x, y = symmetrize_data(x, y)

    # Shift minimum to zero reference
    y = y - y[np.argmin(np.abs(x))]

    popt, _ = curve_fit(
        U_anharm,
        x,
        y,
        bounds=([0, 0], [np.inf, np.inf])
    )

    a, b = popt
    xfit = np.linspace(min(x), max(x), 1000)
    yfit = U_anharm(xfit, a, b)

    x0, barrier = extract_dw_properties(a, b)

    return x, y, xfit, yfit, x0, barrier


# ---------- Main plotting ----------

def plot_frozen_phonons(results, mirror=True):

    qpoints = sorted(results.keys())
    N = len(qpoints)

    fig, axes = plt.subplots(1, N, figsize=(4*N, 4), sharey=True)

    if N == 1:
        axes = [axes]

    color_cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']

    for i, q in enumerate(qpoints):

        ax = axes[i]

        if q == "G":
            ax.set_title(r'$\Gamma$ point')
        else:
            ax.set_title(f"{q} point")

        color_idx = 0

        for g_id, modes in results[q].items():
            for mode_id, data in modes.items():

                df = data["df"]

                x = df["Amplitude"].to_numpy()
                y = df["Energy"].to_numpy() * 1000  # meV

                try:
                    x, y, xfit, yfit, x0, barrier = fit_double_well(x, y)
                except RuntimeError:
                    continue  # skip bad fits

                color = color_cycle[color_idx % len(color_cycle)]
                color_idx += 1

                if mirror:
                    ax.plot(xfit, yfit, '-', color=color)
                else:
                    mask = xfit >= 0
                    ax.plot(xfit[mask], yfit[mask], '-', color=color)

                # Mark origin
                ax.plot(0, 0, 'k.', markersize=4)

                # Annotate physics
                ax.text(
                    0.05,
                    0.9 - 0.1 * color_idx,
                    f"g{g_id} m{mode_id}\n"
                    f"$Q_0$={x0:.2f}\n"
                    f"$\Delta E$={barrier:.1f} meV",
                    transform=ax.transAxes,
                    fontsize=8,
                    color=color
                )

        ax.set_xlabel(r'$Q$ (amu$^{1/2}$Å)')

        if i == 0:
            ax.set_ylabel(r'$\Delta E$ (meV/u.c.)')

        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
from scipy.optimize import curve_fit

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit


# Double well potentials
def U_harm(x, omega):
    return -1/2 * omega * x**2


def U_anharm(x, a, b):
    return -1/2 * a * x**2 + 1/4 * b * x**4


def plotDWs(formula, ids, bulk=True, Ncells=1, mirror=False):

    if isinstance(ids, str):
        ids = [ids]

    ytickmarks = np.arange(-10, 30, 5)

    # ---------- Structure path ----------
    if bulk:
        struc = f'bulk/{formula}'
    else:
        struc = f'slab/{formula}/{Ncells}uc'

    # ---------- Determine q-points ----------
    first_dir = os.path.join('results', struc, ids[0], 'frozen')

    q_points = [
        d for d in os.listdir(first_dir)
        if os.path.isdir(os.path.join(first_dir, d))
    ]

    N = len(q_points)

    # ---------- Colors ----------
    color_cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']
    colors = {id_: color_cycle[i % len(color_cycle)] for i, id_ in enumerate(ids)}

    # ---------- Create figure ----------
    fig, axes = plt.subplots(1, N, figsize=(3*N, 5), sharey='col')

    if N == 1:
        axes = [axes]

    PlotSettings().set_size(fig)
    plt.subplots_adjust(wspace=0.08)

    # ---------- Helper functions ----------

    def load_dw_data(id_, q):
        """Load all DW data for a given calculation id and q-point."""
        dir_q = os.path.join('results', struc, id_, 'frozen', q)

        if not os.path.exists(dir_q):
            return []

        datasets = []

        for mode in os.listdir(dir_q):

            dir_mode = os.path.join(dir_q, mode)

            if not os.path.isdir(dir_mode):
                continue

            try:
                with open(os.path.join(dir_mode, 'freq.txt')) as f:
                    label = f.read().strip()
            except FileNotFoundError:
                label = mode

            for deg in os.listdir(dir_mode):

                dir_deg = os.path.join(dir_mode, deg)

                if not os.path.isdir(dir_deg):
                    continue

                df = pd.read_csv(os.path.join(dir_deg, 'energies.csv'))

                datasets.append((label, df))

        return datasets


    def get_fit(df):

        x = df['Amplitude'].to_numpy()
        y = df['Energy'].to_numpy() * 1000

        y0 = df[df['Amplitude'] == 0]['Energy'].iloc[0] * 1000
        y = y - y0

        xfit = np.linspace(0, max(x), 1000)

        popt, _ = curve_fit(U_anharm, x, y)

        yfit = U_anharm(xfit, *popt)

        return x, y, xfit, yfit


    def plot_curve(ax, x, y, xfit, yfit, color, label=None):

        if mirror:
            xfit = np.concatenate((-xfit[::-1], xfit))
            yfit = np.concatenate((yfit[::-1], yfit))

        #ax.plot(x, y, '.', markersize=4, color=color, label=label)
        ax.plot(xfit, yfit, '-', lw=1, color=color, label=label)

        ax.plot(0, 0, 'r.', markersize=4)

        dx = 0.5 if max(xfit) < 2 else 1
        xtickmarks = np.arange(0, max(xfit), dx)

        xticklabels = np.where(
            xtickmarks % 1 == 0,
            xtickmarks.astype(int).astype(str),
            xtickmarks.astype(str)
        )

        ax.set_xticks(xtickmarks, xticklabels)
        ax.set_yticks(ytickmarks, ytickmarks.astype(str))

        ax.set_xlim(min(xfit), max(xfit))
        ax.set_ylim(-20, 25)

        PlotSettings().set_style_ax(ax, gridlines=True)

    # ---------- Main plotting loop ----------

    for i, q in enumerate(q_points):

        ax = axes[i]

        if q == 'G':
            ax.set_title(r'$\Gamma$ point')
        else:
            ax.set_title(f'{q} point')

        for id_ in ids:

            datasets = load_dw_data(id_, q)

            for j, (label, df) in enumerate(datasets):

                x, y, xfit, yfit = get_fit(df)

                plot_curve(
                    ax,
                    x, y,
                    xfit, yfit,
                    colors[id_],
                    label=f'{id_}: {label}' if j == 0 else None
                )

        ax.legend(loc='upper center')

        ax.set_xlabel(f'$Q_{i+1}$ (amu$^{{1/2}}$Å)')

        if i != 0:
            ax.set_yticklabels([])

    axes[0].set_ylabel(r'$\Delta$ Energy (meV/u.c.)')

    axes[-1].tick_params(axis='y', labelright=True, labelleft=False)

    plt.show()

In [ ]:
plot_frozen_phonons(results, mirror=False)

In [ ]:
plotDWs('BaTiO3', ['0082', 'ML'])

In [ ]:
def plot_dispersion(phonons, vals, width=1, bulk=True, pDOS=False):
    """Function to plot the phonon dispersion and DOS together.
    Parameters:
    - phonons: List of Phonopy objects containing phonon data.
    - vals: List of labels for each phonon object.
    - width: Width of the figure as a fraction of the total width.
    - pDOS: Whether to plot the projected density of states (PDOS) (default is True).
    - bulk: Boolean indicating if the system is bulk (True) or slab (False).
    Returns:
    - None. The function creates a plot of the phonon dispersion and DOS.
    """

    # Define tickmarks for the x- and y-axis
    E_tickmarks = np.arange(-10, 26, 5)
    # Convert tickmarks to strings with i for negative numbers
    E_ticklabels = []
    for tick in E_tickmarks:
        if tick < 0:
            E_ticklabels.append(f'{abs(tick)}i')
        else:
            E_ticklabels.append(f'{tick}')

    # Define tickmarks for the x- and y-axis
    dos_tickmarks = np.arange(0, 7, 1)

    colors = ["black", "blue", "red", "purple", "orange", "green"]

    # Make a simple figure where graphs are plotted
    fig, axes = plt.subplots(1, 2, figsize=(9.6, 5),
                             sharey='col', gridspec_kw={'width_ratios': [1, 0.4]})
    
    # Define two axes, one for the band structure and one for the DOS
    ax1, ax2 = axes
    
    PlotSettings().set_size(fig, width)
    plt.subplots_adjust(wspace=0.08)
    
    def _plot_disp(ax, phonon, val, col='k', vlines=True):
        # Extract phonon dispersion data
        (dist, X, freq, labels) = get_phonon_dispersion(phonon, bulk)
        dist = np.array(dist)
        dist /= dist[-1][-1]  # Normalize distances to the total length of the path
        X /= X[-1]  # Normalize high symmetry point locations to the total length of the path
        if vlines:
            # Plot vertical lines at symmetry points
            ax.vlines(X, E_tickmarks[0], E_tickmarks[-1], color='0.5', lw=0.8)
        # Plot dashed line at 0
        #ax.axhline(y=0, color='k', linestyle=':')
        # Determine the number of segments between symmetry points and the number of modes
        n_segments = len(freq)
        n_modes = freq[0].shape[1]
        # Loop over all segments and modes and plot everything
        for i in range(n_segments):
            for j in range(n_modes):
                if i == 0 and j == 0:
                    ax.plot(dist[i], freq[i][:, j], color=col, lw=1, label=f'{val}')
                else:
                    ax.plot(dist[i], freq[i][:, j], color=col, lw=1)
        # Set x- and y-ticks
        ax.set_xticks(X, labels)
        ax.set_yticks(E_tickmarks, E_ticklabels)
        # Set x- and y-limits
        ax.set_xlim(X[0], X[-1])
        ax.set_ylim(E_tickmarks[0], E_tickmarks[-1])

    def _plot_dos(ax, phonon, val='DOS', col='k'):
        # Extract total DOS data
        (dosx, dosy) = get_phonon_dos(phonon, bulk)
        # Plot total DOS
        ax.plot(dosx, dosy, lw=1, color=col, label=f'{val}')
        if pDOS:
            ax.fill_between(dosx, dosy, color='lightgray', alpha=0.5)

    def _plot_pdos(ax, phonon):
        atom_colors = {'Ba': 'tab:blue', 'Sr': 'tab:purple',
                       'Ti': 'tab:orange', 'O': 'tab:red'}
        # Extract PDOS data
        (pdosx, pdosy, symbols) = get_phonon_pdos(phonon, bulk)
        # Plot PDOS
        for i in range(pdosx.shape[0]):
            ax.plot(pdosx[i], pdosy, lw=1, color=atom_colors[symbols[i]], label=f'{symbols[i]}')
        # Get all handles and labels
        handles, labels = ax.get_legend_handles_labels()
        # Remove duplicates and sort for the legend
        sorted_handles, sorted_labels = order_labels(symbols, handles, labels)
        # Add legend with duplicates removed and sorted labels
        ax.legend(sorted_handles, sorted_labels, loc='best', fontsize=14)

    # Plot dashed line at Fermi level for both subplots
    ax1.axhline(y=0, color='k', linestyle=':', lw=0.8)
    ax2.axhline(y=0, color='k', linestyle=':', lw=0.8)
    for i, phonon in enumerate(phonons):
        # Plot phonon dispersion and total DOS for PW
        _plot_disp(ax1, phonon, val=vals[i], col=colors[i])
        _plot_dos(ax2, phonon, val=vals[i], col=colors[i])
        if pDOS:
            # Plot PDOS for PW
            _plot_pdos(ax2, phonon)

    # Set x- and y-label
    ax1.set_xlabel('k-points')
    ax1.set_ylabel('Frequency (THz)')
    # Add minor tickmarks to the y-axis
    ax1.yaxis.set_minor_locator(AutoMinorLocator())
    
    ax2.set_xlabel('DOS (1/THz)')
    ax2.legend(loc='upper right')
    # Force x- and y-ticks
    ax2.set_xticks(dos_tickmarks, dos_tickmarks)
    ax2.set_yticks(E_tickmarks, E_ticklabels)
    # Set limits to match
    ax2.set_xlim(dos_tickmarks[0], dos_tickmarks[-1])
    ax2.set_ylim(E_tickmarks[0], E_tickmarks[-1])
    # Hide y-tick labels
    ax2.set_yticklabels([])
    
    # Show figure
    plt.show()

In [ ]:
plot_dispersion([phonon_LCAO, phonon_ML], ['LCAO', 'ML'], width=1, bulk=True, pDOS=False)